In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots

include("functions.jl")
Random.seed!(2025)
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000

Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]

tau = length(Istar_obs)

model_tag_sym = :sliding
KMAX_UPPER = 30  
KMAX_fixed = 26

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 2

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER,
            k_max_fixed = KMAX_fixed)
    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [sliding_model] Fitting chain 2 (tau=34)
[ Info: [sliding] iter 1000/1000000 elapsed=3.7s, rate=0.080, mean=[1.106, 0.00029, 1.182], std=[0.0696, 0.000415, 0.0618] [ADAPT]
[ Info: [sliding] iter 2000/1000000 elapsed=6.9s, rate=0.053, mean=[1.483, 0.00020, 1.299], std=[0.4121, 0.000311, 0.1221] [ADAPT]
[ Info: [sliding] iter 3000/1000000 elapsed=9.1s, rate=0.041, mean=[1.770, 0.00016, 1.376], std=[0.4957, 0.000264, 0.1393] [ADAPT]
[ Info: [sliding] iter 4000/1000000 elapsed=11.3s, rate=0.033, mean=[1.933, 0.00014, 1.422], std=[0.4979, 0.000235, 0.1406] [ADAPT]
[ Info: [sliding] iter 5000/1000000 elapsed=13.5s, rate=0.029, mean=[2.059, 0.00013, 1.451], std=[0.4995, 0.000215, 0.1362] [ADAPT]
[ Info: [sliding] iter 6000/1000000 elapsed=15.7s, rate=0.025, mean=[2.145, 0.00012, 1.470], std=[0.4873, 0.000201, 0.1299] [ADAPT]
[ Info: [sliding] iter 7000/1000000 elapsed=17.9s, rate=0.022, mean=[2.195, 0.00011, 1.482], std=[0.4645, 0.000189, 0.1231] [ADAPT]
[ Info: [sliding] iter 8000/10